# Bài 4: DML — UPDATE, DELETE, MERGE (Upsert) & SCD Type 2

## Mục tiêu
- Thực hiện `UPDATE`, `DELETE` trực tiếp trên bảng Delta (điều Parquet/Hive thuần không hỗ trợ tốt).
- Làm chủ `MERGE INTO` để upsert (insert-or-update), bao gồm cả `WHEN NOT MATCHED BY SOURCE`.
- Hiểu và cài đặt mẫu **Slowly Changing Dimension Type 2 (SCD2)** bằng `MERGE`.


## 4.1. UPDATE / DELETE

Khác với Parquet/Hive thuần (phải đọc toàn bộ, lọc, rồi ghi đè cả bảng), Delta Lake hỗ trợ `UPDATE`/`DELETE` như một bảng quan hệ thật:

```sql
UPDATE db.tbl SET status = 'shipped' WHERE order_id = 5;
DELETE FROM db.tbl WHERE order_date < '2020-01-01';
```

Cơ chế bên dưới: Delta **không sửa file Parquet tại chỗ** (immutable). Với mỗi file Parquet có ít nhất 1 dòng thoả điều kiện, Delta đọc file đó, ghi ra **file Parquet mới** với dữ liệu đã cập nhật/xoá, rồi trong transaction log: `remove` file cũ + `add` file mới. Các file hoàn toàn không bị ảnh hưởng thì giữ nguyên, không đọc/ghi lại.

Cũng có Python API tương đương qua `DeltaTable`:
```python
from delta.tables import DeltaTable
dt = DeltaTable.forName(spark, "db.tbl")
dt.update(condition="order_id = 5", set={"status": "'shipped'"})
dt.delete("order_date < '2020-01-01'")
```

## 4.2. MERGE INTO (Upsert)

`MERGE` so khớp bảng đích với 1 nguồn dữ liệu mới theo điều kiện, rồi áp dụng hành động khác nhau cho từng nhóm:

```sql
MERGE INTO target t
USING source s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE   -- (tuỳ chọn) xoá record có trong target nhưng không có trong source
```

- `WHEN MATCHED` — record khớp cả 2 bên → có thể `UPDATE` (toàn bộ `SET *` hoặc chọn cột) hoặc `DELETE`.
- `WHEN NOT MATCHED [BY TARGET]` — record chỉ có ở source → `INSERT`.
- `WHEN NOT MATCHED BY SOURCE` — record chỉ có ở target (Delta 2.3+) → thường dùng để `DELETE` (đồng bộ hoá "source of truth" đầy đủ).
- Có thể thêm điều kiện phụ: `WHEN MATCHED AND s.deleted = true THEN DELETE`.
- Nhiều `WHEN MATCHED`/`WHEN NOT MATCHED` được đánh giá **theo thứ tự khai báo**, action đầu tiên thoả điều kiện sẽ được áp dụng.

Đây là công cụ trung tâm để xây **incremental pipeline** (CDC ingest, dedup, upsert từ nguồn OLTP...).

## 4.3. SCD Type 2 bằng MERGE

SCD2: khi 1 thuộc tính của dimension thay đổi, **giữ lại bản ghi cũ** (đánh dấu hết hiệu lực) và **thêm bản ghi mới**, thay vì ghi đè. Cần các cột `effective_date`, `end_date`, `is_current`. Cách làm chuẩn: `MERGE` 2 bước conceptual, thực hiện bằng **UNION source với 1 cờ `merge_key` đặc biệt** để vừa "đóng" bản ghi cũ vừa "mở" bản ghi mới trong 1 lệnh MERGE duy nhất (xem ví dụ).


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai04"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai04-dml")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 4.4. Ví dụ minh hoạ

In [ ]:
spark.sql("DROP TABLE IF EXISTS bai04.inventory")
rows = [(1, "Keyboard", 100), (2, "Mouse", 200), (3, "Monitor", 15)]
spark.createDataFrame(rows, ["product_id", "name", "stock"]).write.format("delta").saveAsTable("bai04.inventory")

spark.sql("UPDATE bai04.inventory SET stock = stock - 10 WHERE product_id = 1")
spark.sql("DELETE FROM bai04.inventory WHERE stock < 20")
spark.sql("SELECT * FROM bai04.inventory ORDER BY product_id").show()


In [ ]:
# MERGE INTO: upsert du lieu moi vao bang inventory
updates = spark.createDataFrame(
    [(1, "Keyboard", 500), (4, "Webcam", 50)],   # id=1 da ton tai (update), id=4 la moi (insert)
    ["product_id", "name", "stock"],
)
updates.createOrReplaceTempView("inventory_updates")

spark.sql("""
MERGE INTO bai04.inventory t
USING inventory_updates s
ON t.product_id = s.product_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * FROM bai04.inventory ORDER BY product_id").show()


In [ ]:
# WHEN NOT MATCHED BY SOURCE: dong bo hoa day du - xoa nhung record khong con trong source
full_snapshot = spark.createDataFrame([(1, "Keyboard", 500), (4, "Webcam", 50)], ["product_id", "name", "stock"])
full_snapshot.createOrReplaceTempView("inventory_full_snapshot")

spark.sql("""
MERGE INTO bai04.inventory t
USING inventory_full_snapshot s
ON t.product_id = s.product_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
spark.sql("SELECT * FROM bai04.inventory ORDER BY product_id").show()
# product_id=2 (Mouse) khong con trong snapshot -> bi xoa khoi bang dich


In [ ]:
# SCD Type 2: theo doi lich su thay doi gia san pham
spark.sql("DROP TABLE IF EXISTS bai04.dim_product_scd2")
spark.sql("""
CREATE TABLE bai04.dim_product_scd2 (
    product_id INT, name STRING, price DOUBLE,
    effective_date DATE, end_date DATE, is_current BOOLEAN
) USING DELTA
""")
spark.sql("""
INSERT INTO bai04.dim_product_scd2 VALUES
  (1, 'Keyboard', 25.0, DATE'2024-01-01', DATE'9999-12-31', true),
  (2, 'Mouse',    10.0, DATE'2024-01-01', DATE'9999-12-31', true)
""")
spark.sql("SELECT * FROM bai04.dim_product_scd2 ORDER BY product_id").show()


In [ ]:
# Gia Keyboard doi tu 25.0 -> 30.0 ke tu 2024-03-01. Ap dung SCD2 bang 1 lenh MERGE duy nhat
from pyspark.sql import functions as F

changes = spark.createDataFrame([(1, "Keyboard", 30.0, "2024-03-01")], ["product_id", "name", "price", "effective_date"])

# "merge_key": voi record thay doi -> tao 2 dong trong source (1 de UPDATE dong cu, 1 de INSERT dong moi)
staged_updates = (
    changes.alias("c")
    .join(
        spark.table("bai04.dim_product_scd2").filter("is_current = true").alias("t"),
        on="product_id",
    )
    .filter("t.price <> c.price")   # chi xu ly khi thuc su co thay doi
    .select("c.product_id", "c.name", "c.price", "c.effective_date")
)

# dong "close cu" dung merge_key = product_id that; dong "insert moi" dung merge_key = NULL de khong match
close_old = staged_updates.withColumn("mergeKey", F.col("product_id"))
insert_new = staged_updates.withColumn("mergeKey", F.lit(None).cast("int"))
staged = close_old.unionByName(insert_new)
staged.createOrReplaceTempView("scd2_staged")

spark.sql("""
MERGE INTO bai04.dim_product_scd2 t
USING scd2_staged s
ON t.product_id = s.mergeKey AND t.is_current = true
WHEN MATCHED THEN
  UPDATE SET t.end_date = date_sub(s.effective_date, 1), t.is_current = false
WHEN NOT MATCHED THEN
  INSERT (product_id, name, price, effective_date, end_date, is_current)
  VALUES (s.product_id, s.name, s.price, s.effective_date, DATE'9999-12-31', true)
""")
spark.sql("SELECT * FROM bai04.dim_product_scd2 ORDER BY product_id, effective_date").show()


## 4.5. Thực hành

**Bài 1** — Tạo bảng `bai04.accounts (account_id INT, balance DOUBLE)` với 4 dòng. Dùng `UPDATE` để cộng thêm 100 vào balance của 1 account cụ thể. Dùng `DELETE` để xoá các account có `balance < 0`.

**Bài 2** — Tạo 1 DataFrame "nguồn" gồm 2 account đã tồn tại (update) và 2 account mới (insert). Viết `MERGE INTO` để upsert vào `bai04.accounts`.

**Bài 3** — Thêm `WHEN MATCHED AND ... THEN DELETE` vào câu MERGE ở Bài 2: nếu source có cột `closed = true` thì xoá luôn account đó khỏi bảng đích thay vì update.

**Bài 4** — Dùng `WHEN NOT MATCHED BY SOURCE THEN DELETE` để đồng bộ `bai04.accounts` với 1 "full snapshot" chỉ gồm 2 account — các account không có trong snapshot phải biến mất khỏi bảng đích.

**Bài 5 (nâng cao)** — Áp dụng đúng mẫu SCD2 ở phần ví dụ cho bảng dimension của riêng bạn (ví dụ `dim_customer` với cột `address` thay đổi theo thời gian). Sau khi chạy MERGE, viết 1 câu SQL lấy ra **giá trị đang hiệu lực tại một thời điểm bất kỳ trong quá khứ** (không phải "hiện tại") bằng cách filter theo `effective_date`/`end_date`.


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

In [ ]:
# TODO: Bài 4


### Vùng làm bài — Bài 5

In [ ]:
# TODO: Bài 5


---
## Gợi ý / đáp án tham khảo

In [ ]:
# Dap an Bai 1
spark.sql("DROP TABLE IF EXISTS bai04.accounts")
spark.createDataFrame([(1,100.0),(2,-5.0),(3,50.0),(4,-20.0)], ["account_id","balance"]) \
    .write.format("delta").saveAsTable("bai04.accounts")
spark.sql("UPDATE bai04.accounts SET balance = balance + 100 WHERE account_id = 1")
spark.sql("DELETE FROM bai04.accounts WHERE balance < 0")
spark.sql("SELECT * FROM bai04.accounts ORDER BY account_id").show()


In [ ]:
# Dap an Bai 2
src = spark.createDataFrame([(1, 999.0), (3, 60.0), (5, 10.0), (6, 20.0)], ["account_id", "balance"])
src.createOrReplaceTempView("accounts_src")
spark.sql("""
MERGE INTO bai04.accounts t USING accounts_src s
ON t.account_id = s.account_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * FROM bai04.accounts ORDER BY account_id").show()


In [ ]:
# Dap an Bai 3
src2 = spark.createDataFrame([(1, 999.0, False), (3, 0.0, True), (7, 15.0, False)], ["account_id", "balance", "closed"])
src2.createOrReplaceTempView("accounts_src2")
spark.sql("""
MERGE INTO bai04.accounts t USING accounts_src2 s
ON t.account_id = s.account_id
WHEN MATCHED AND s.closed = true THEN DELETE
WHEN MATCHED THEN UPDATE SET t.balance = s.balance
WHEN NOT MATCHED THEN INSERT (account_id, balance) VALUES (s.account_id, s.balance)
""")
spark.sql("SELECT * FROM bai04.accounts ORDER BY account_id").show()


In [ ]:
# Dap an Bai 4
snapshot = spark.createDataFrame([(1, 999.0), (5, 10.0)], ["account_id", "balance"])
snapshot.createOrReplaceTempView("accounts_full_snapshot")
spark.sql("""
MERGE INTO bai04.accounts t USING accounts_full_snapshot s
ON t.account_id = s.account_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
spark.sql("SELECT * FROM bai04.accounts ORDER BY account_id").show()
# Chi con account_id 1 va 5 - cac account khac bi xoa vi khong co trong snapshot


**Đáp án Bài 5**: Áp dụng cùng mẫu "close cũ + insert mới" bằng `unionByName` và `mergeKey` như ví dụ ở mục 4.4, thay `dim_product_scd2` bằng `dim_customer_scd2` (cột `address` thay cho `price`). Truy vấn "giá trị đang hiệu lực tại thời điểm X":

```sql
SELECT * FROM bai04.dim_customer_scd2
WHERE customer_id = 1
  AND effective_date <= DATE'2024-02-15'
  AND end_date        >= DATE'2024-02-15'
```

Không dùng `is_current = true` vì đó chỉ đúng cho thời điểm hiện tại — để lấy giá trị lịch sử tại 1 mốc bất kỳ, phải so sánh mốc đó với khoảng `[effective_date, end_date]`.
